In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision import datasets

In [3]:
transform = transforms.Compose([
    transforms.Resize((64,64),antialias = True),
    transforms.ToTensor()
])

from pathlib import Path
from torch.utils.data import Subset

root = Path("/kaggle/input/datasets/drgfreeman/rockpaperscissors")

class RockPaperScissors(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = ["paper", "rock", "scissors"]

        for name in classes:
            if not (Path(directory) / name).is_dir():
                raise FileNotFoundError(f"Missing class folder: {name}")

        return classes, {name: i for i, name in enumerate(classes)}

dataset = RockPaperScissors(root=root, transform=transform)

print(dataset.class_to_idx)
print("Total images:", len(dataset))

{'paper': 0, 'rock': 1, 'scissors': 2}
Total images: 2188


In [4]:
total= len(dataset)

train_size = int(0.7*total)
val_size = int(0.15*total)
test_size = total - train_size - val_size

trainset, valset, testset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator  = torch.Generator().manual_seed(42)
)

In [5]:
batch_size = 64

training_data = DataLoader(trainset, batch_size = batch_size, shuffle= True)

val_data = DataLoader(valset, batch_size = batch_size, shuffle= True)

testing_data = DataLoader(testset, batch_size = batch_size, shuffle= True)


**CNN 1**

In [6]:
class CNN1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 5)
        self.pool1 = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,5,5)
        self.fc1 = nn.Linear(5*13*13, 117)
        self.fc2 = nn.Linear(117,3)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool1(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [8]:
CNN = CNN1()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(CNN.parameters(), lr = 0.001)
for epoch in range(10):
    CNN.train()
    running_loss = 0
    total_images = 0
    for i, data in enumerate(training_data, 0):
        inputs,labels = data

        optimizer.zero_grad()

        outputs = CNN(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        total_images += batch_size

    epoch_loss = running_loss / total_images
    print(f"Epoch {epoch + 1}/10 | Training loss: {epoch_loss:.4f}")

    CNN.eval()
    running_val_loss = 0
    total_val_images = 0
    with torch.no_grad(): 
        for data in val_data:
            inputs, labels = data
            predictions = CNN(inputs)
            batch_loss = criterion(predictions, labels)

            batch_size = inputs.size(0)
            running_val_loss += batch_loss.item()*batch_size
            total_val_images += batch_size

        epoch_val_loss = running_val_loss/total_val_images
        print(f"Epoch {epoch + 1}/10 | Validation loss: {epoch_val_loss:.4f}")

            

print("Finished Training")

Epoch 1/10 | Training loss: 1.0810
Epoch 1/10 | Validation loss: 1.0205
Epoch 2/10 | Training loss: 0.8211
Epoch 2/10 | Validation loss: 0.5953
Epoch 3/10 | Training loss: 0.5291
Epoch 3/10 | Validation loss: 0.4560
Epoch 4/10 | Training loss: 0.4426
Epoch 4/10 | Validation loss: 0.4667
Epoch 5/10 | Training loss: 0.3838
Epoch 5/10 | Validation loss: 0.3770
Epoch 6/10 | Training loss: 0.3243
Epoch 6/10 | Validation loss: 0.4004
Epoch 7/10 | Training loss: 0.2687
Epoch 7/10 | Validation loss: 0.2983
Epoch 8/10 | Training loss: 0.2165
Epoch 8/10 | Validation loss: 0.2489
Epoch 9/10 | Training loss: 0.1974
Epoch 9/10 | Validation loss: 0.2494
Epoch 10/10 | Training loss: 0.1627
Epoch 10/10 | Validation loss: 0.2052
Finished Training


In [10]:
CNN.eval()
running_test_loss = 0
total_test_images = 0
with torch.no_grad(): 
    for data in testing_data:
        inputs, labels = data
        predictions = CNN(inputs)
        batch_loss = criterion(predictions, labels)

        batch_size = inputs.size(0)
        running_test_loss += batch_loss.item()*batch_size
        total_test_images += batch_size

    epoch_test_loss = running_test_loss/total_test_images
    print(f"Epoch {epoch + 1}/10 | Test loss: {epoch_test_loss:.4f}")


Epoch 10/10 | Test loss: 0.2366


**CNN 2**

In [ ]:
class CNN2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels = 3, out_channels = 3, kernel_size = 5, groups =3. bias = False),
            nn.Conv2d(in_channels = 3, out_channels = 6, kernel_size = 1)
        )
    